In [1]:
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (469 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [12]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [13]:
import subprocess
import time

subprocess.Popen(["ollama", "serve"])
time.sleep(3)
print("Ollama başlatıldı!")

Ollama başlatıldı!


In [14]:
!ollama pull llama3.1:latest


In [ ]:
from datasets import load_dataset
meetingbank = load_dataset("huuuyeah/meetingbank")

train_data = meetingbank['train']
test_data = meetingbank['test']
val_data = meetingbank['validation']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.json:   0%|          | 0.00/88.4M [00:00<?, ?B/s]

validation.json:   0%|          | 0.00/13.2M [00:00<?, ?B/s]

test.json:   0%|          | 0.00/13.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5169 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/861 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/862 [00:00<?, ? examples/s]

In [ ]:
def generator(data_split):
  for instance in data_split:
    yield instance['id'], instance['summary'], instance['transcript']

In [ ]:
import requests
import json
import csv
import time
import pandas as pd

SYSTEM_PROMPT = """You are a meeting assistant. Given a meeting transcript, produce a JSON object with two keys:
1. "summary": A concise, well-structured meeting summary in paragraph form. Cover the key discussion points, decisions made, and conclusions.
2. "action_items": An array of action items extracted from the meeting. Each item is an object with:
   - "description": A clear, actionable description of what needs to be done.
   - "assignee": The name of the person responsible. ONLY use names that actually appear as speakers in the transcript. Set to null if no specific person was assigned.
IMPORTANT: Do NOT invent or hallucinate assignee names. Only use speaker names that appear in the transcript.
Return ONLY valid JSON, no markdown fences, no extra text. Example format:
{"summary": "The team discussed...", "action_items": [{"description": "Task description here", "assignee": null}]}"""


def generate_summary(transcript):
    prompt = f"{SYSTEM_PROMPT}\n\nTranscript:\n{transcript}"

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.1",
            "prompt": prompt,
            "stream": False
        }
    )

    raw = response.json()["response"].strip()

    try:
        parsed = json.loads(raw)
        return parsed
    except json.JSONDecodeError:
        return raw


# tutanaklar.csv'yi oku
df = pd.read_csv("/tutanaklar.csv")

output_file = "summaries_turkish.csv"

with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["session", "full_text", "reference_summary", "generated_response"])
    writer.writeheader()

    for i, row in df.iterrows():
        transcript = str(row["full_text"])
        reference_summary = str(row["summary"])
        session = str(row["session"])

        print(f"Processing session: {session} ({i+1}/{len(df)})...")

        generated_response = generate_summary(transcript)

        writer.writerow({
            "session": session,
            "full_text": transcript,
            "reference_summary": reference_summary,
            "generated_response": json.dumps(generated_response, ensure_ascii=False)
        })

        time.sleep(0.5)

print(f"Tamamlandı! '{output_file}' dosyasına kaydedildi.")

FileNotFoundError: [Errno 2] No such file or directory: '/tutanaklar.csv'

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!pip install deepeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 844.6/844.6 kB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.9/226.9 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 4.3 MB/s eta 0:00:00


In [15]:
!pip install ollama

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-1' coro=<wait_for() done, defined at /usr/lib/python3.12/asyncio/tasks.py:472> exception=ConnectionError('Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download')>
Traceback (most recent call last):
  File "/usr/lib/python3.12/asyncio/tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/tasks.py", line 520, in wait_for
    return await fut
           ^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/deepeval/tracing/tracing.py", line 1323, in async_wrapper
    return await func(*args, **func_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/deepeval/metrics/g_eval/g_eval.py", line 191, in a_measure
    await self._a_generate_evaluation_steps(multimodal)
  File "/usr/local/lib/python3.12/dis

In [16]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
import pandas as pd
import json
from deepeval.models import OllamaModel



In [17]:
import os
import time
import json
import pandas as pd
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.models import OllamaModel

In [ ]:
import os
import time
import json
import pandas as pd
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.models import OllamaModel

os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"

df = pd.read_csv("/content/summaries_turkish.csv")

MEETING_ASSESSMENT_QUESTIONS = [
    "Does the summary contain any information not present in the transcript?",
    "Does the summary contradict anything stated in the transcript?",
    "Are the main topics of the meeting captured in the summary?",
    "If any action items or next steps were discussed, are they reflected in the summary?",
]

metric = SummarizationMetric(
    threshold=0.3,
    model=OllamaModel(model="llama3.1"),
    verbose_mode=True,
    assessment_questions=MEETING_ASSESSMENT_QUESTIONS,
    n=0
)

results = []

for i, row in df.iterrows():
    try:
        parsed_content = json.loads(row["generated_response"])
        if isinstance(parsed_content, dict):
            generated_summary = parsed_content.get("summary", str(parsed_content))
        else:
            generated_summary = str(parsed_content)
    except (json.JSONDecodeError, TypeError):
        generated_summary = str(row["generated_response"])

    test_case = LLMTestCase(
        input=row["full_text"],        # transcript → full_text
        actual_output=generated_summary,
    )

    try:
        metric.measure(test_case)
        results.append({
            "session": row["session"],
            "score": metric.score,
            "reason": metric.reason,
            "passed": metric.is_successful()
        })
        print(f"[{i+1}/{len(df)}] Session: {row['session']} | Score: {metric.score:.2f} | {metric.reason}")
    except Exception as e:
        results.append({
            "session": row["session"],
            "score": None,
            "reason": str(e),
            "passed": False
        })
        print(f"[{i+1}/{len(df)}] HATA (session: {row['session']}): {e}")

    time.sleep(3)

results_df = pd.DataFrame(results)
results_df.to_csv("summarization_results_turkish.csv", index=False)
print(f"\nOrtalama Skor: {results_df['score'].mean():.2f}")
print(f"Başarılı: {results_df['passed'].sum()}/{len(df)}")

Output()

**************************************************

Summarization Verbose Logs

**************************************************

Truths (limit=None):
[
    "Açýlma Saati: 14.00",
    "Kapanma Saati: 22.20"
] 
 
Claims:
[
    "Bu metin, Türkiye Büyük Millet Meclisi (TBMM) oturumunun bir bölümünü içeriyor.",
    "TBMM'nin 82'nci Birleşiminin Altıncı Oturumu'nun açılışı ve bu oturma sırasında tartışılan kanun teklifleri 
hakkında bilgi verilmektedir.",
    "Görüşülen konular arasında, 'Sosyal Yardım Veri Tabanı Kanunu' ve 'Sosyal Ağ Sağlayıcısı Düzenlemesi' gibi 
önemli konu başlıkları yer almaktadır.",
    "Bu konularda hükûmet yetkilileri ve milletvekilleri arasındaki tartışmalar hakkında bilgi verilmektedir.",
    "Genel olarak, TBMM'nin 82'nci Birleşiminin Altıncı Oturumu'nda kanun teklifleri ve diğer konular hakkında 
discussionlar yapılmakta ve fikirlerin değiştirilmeye çalışılmaktadır."
] 
 
Assessment Questions:
[
    "Does the summary contain any information not present in the transcript?",
    "Does the summary contradict anything stated in the transcript?",
    "Are the main topics of the meeting captured in the summary?",
    "If any action items or next steps were discussed, are they reflected in the summary?"
] 
 
Coverage Verdicts:
[
    {
        "summary_verdict": "no",
        "original_verdict": "Evet",
        "question": "Does the summary contain any information not present in the transcript?"
    },
    {
        "summary_verdict": "no",
        "original_verdict": "Hay\u0131r",
        "question": "Does the summary contradict anything stated in the transcript?"
    },
    {
        "summary_verdict": "yes",
        "original_verdict": "Evet",
        "question": "Are the main topics of the meeting captured in the summary?"
    },
    {
        "summary_verdict": "no",
        "original_verdict": "Hay\u0131r",
        "question": "If any action items or next steps were discussed, are they reflected in the summary?"
    }
] 
 
Alignment Verdicts:
[
    {
        "verdict": "idk",
        "reason": "The original text does not mention the summary claims about TBMM's 82nd Session, so we cannot 
determine if they are correct or not."
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "no",
        "reason": "The original text only mentions 'Sosyal Yard\u0131m Veri Taban\u0131 Kanunu' and does not 
mention 'Sosyal A\u011f Sa\u011flay\u0131c\u0131s\u0131 D\u00fczenlemesi'."
    },
    {
        "verdict": "idk",
        "reason": "The original text does not provide information about the discussions between government 
officials and parliamentarians, so we cannot determine if the summary claims are correct or not."
    },
    {
        "verdict": "no",
        "reason": "The original text states that the session started at 14:00 and ended at 22:20, but the summary 
claim mentions 'discussionlar yap\u0131lmakta ve fikirlerin de\u011fi\u015ftirilmeye 
\u00e7al\u0131\u015f\u0131lmaktad\u0131r', which implies a more dynamic process than what is described in the 
original text."
    }
]
 
Score: 0
Reason: The score is 0.00 because the summary contains contradicting information with the original text, such as 
mentioning 'Sosyal Ağ Sağlayıcısı Düzenlemesi' which is not present in the original text, and implying a more 
dynamic process than described, while also adding extra information not mentioned in the original text about TBMM's
82nd Session and discussions between government officials and parliamentarians.

...

Summarization Verbose Logs

**************************************************

Truths (limit=None):
[
    "28'inci Dönem Dördüncü Yasama Yılı",
    "1 Ekim 2025 Çarşamba günü"
] 
 
Claims:
[
    "Bu metin, Türkiye Büyük Millet Meclisinin (TBMM) Genel Kurulunun açılış konuşmasıdır.",
    "Cumhurbaşkanı Recep Tayyip Erdoğan tarafından verilen konulardan bahseden metinde, Türkiye'nin ekonomik ve 
siyasi gelişmeleri hakkında bilgi verilmektedir.",
    "Cumhurbaşkanı Erdoğan, Türkiye'nin son yirmi üç yılda alınmış büyük mesafeyi Türkiye Yüzyılı'na yakınsayacak 
ve Türkiye'ye dar gelmeyecek yeni bir anayasayla taçlandıracağına inandığını belirtmiştir.",
    "Türkiye'nin imkânlarını, potansiyelini, zenginliklerini daha da artırmak için koşturduğunu ve ülkenin 86 
milyon vatandaşının ihtiyaçları için seferber edildiğini vurgulamıştır.",
    "TBMM'nin gündemindeki işlerin sıralamasında değişikliğe gidileceği ve bazı konuların daha öncelikli olarak ele
alınacağı belirtilmiştir.",
    "Denetim konuları geçirilmeyecek ve gündemin 'Kanun Teklifleri ile Komisyonlardan Gelen Diğer İşler' kısmında 
bulunan işlerin görüştürüleceği, ancak siyasi parti grupları adına yapılacak konuşmaların süresinin en fazla 2 kişi
tarafından kullanılacağı açıklanmıştır.",
    "Genel Kurulun 28'inci Yasama Dönemi Dördüncü Yasama Yılı açılış konuşmasıyla ilgili öneri sunulmuştur ve 
Danışma Kurulu önerisi kabul edilmiştir."
] 
 
Assessment Questions:
[
    "Does the summary contain any information not present in the transcript?",
    "Does the summary contradict anything stated in the transcript?",
    "Are the main topics of the meeting captured in the summary?",
    "If any action items or next steps were discussed, are they reflected in the summary?"
] 
 
Coverage Verdicts:
[
    {
        "summary_verdict": "no",
        "original_verdict": "No",
        "question": "Does the summary contain any information not present in the transcript?"
    },
    {
        "summary_verdict": "no",
        "original_verdict": "No",
        "question": "Does the summary contradict anything stated in the transcript?"
    },
    {
        "summary_verdict": "yes",
        "original_verdict": "Yes",
        "question": "Are the main topics of the meeting captured in the summary?"
    },
    {
        "summary_verdict": "no",
        "original_verdict": "Yes",
        "question": "If any action items or next steps were discussed, are they reflected in the summary?"
    }
] 
 
Alignment Verdicts:
[
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "idk",
        "reason": "The original text does not mention the economic and political developments of Turkey, it only 
mentions that the speech is about these topics."
    },
    {
        "verdict": "no",
        "reason": "The summary claims that President Erdo\u011fan believes in a new constitution that will surpass 
the 'Turkey Century', but the original text states that he believes in a new constitution that will not be 
surpassed by it."
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "idk",
        "reason": "The original text does not mention the prioritization of tasks, it only mentions that some tasks
will be given priority over others."
    },
    {
        "verdict": "no",
        "reason": "The summary claims that denaturation topics will be passed, but the original text states that 
they will not be passed."
    },
    {
        "verdict": "yes",
        "reason": null
    }
]
 
Score: 0.42857142857142855
Reason: The score is 0.43 because the summary contains contradictory information (President Erdoğan's belief on a 
new constitution and denaturation topics) compared to the original text, indicating some inaccuracies. 
Additionally, it includes extra information not mentioned in the original text (economic and political developments
of Turkey), which may indicate a lack of focus or relevance. However, the summary does attempt to capture the main 
points of the speech, including President Erdoğan's belief on a new co

======================================================================

[82/82] Session: 1 . Birleşim | Score: 0.43 | The score is 0.43 because the summary contains contradictory information (President Erdoğan's belief on a new constitution and denaturation topics) compared to the original text, indicating some inaccuracies. Additionally, it includes extra information not mentioned in the original text (economic and political developments of Turkey), which may indicate a lack of focus or relevance. However, the summary does attempt to capture the main points of the speech, including President Erdoğan's belief on a new constitution and prioritization of tasks.

Ortalama Skor: 0.26
Başarılı: 33/82


In [8]:
!pip install rouge-score


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=3f5b60ab03597c10002f99b18396f84f5ffcd6459439290319223017f232ddcf
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
import os
import time
import json
import pandas as pd
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric
from deepeval.models import OllamaModel

os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"

df = pd.read_csv("/content/summaries_turkish.csv")

metric = FaithfulnessMetric(
    threshold=0.7,
    model=OllamaModel(model="llama3.1"),
    verbose_mode=True
)

faithfulness_results = []

for i, row in df.head(82).iterrows():
    try:
        parsed_content = json.loads(row["generated_response"])
        if isinstance(parsed_content, dict):
            generated_summary = parsed_content.get("summary", str(parsed_content))
        else:
            generated_summary = str(parsed_content)
    except (json.JSONDecodeError, TypeError):
        generated_summary = str(row["generated_response"])

    test_case = LLMTestCase(
        input=row["full_text"],
        actual_output=generated_summary,
        retrieval_context=[row["full_text"]]  # FaithfulnessMetric için zorunlu
    )

    try:
        metric.measure(test_case)
        faithfulness_results.append({
            "index": i,
            "score": metric.score,
            "reason": metric.reason,
            "passed": metric.is_successful()
        })
        print(f"[{i+1}/100] Score: {metric.score:.2f} | {metric.reason}")
    except Exception as e:
        faithfulness_results.append({"index": i, "score": None, "reason": str(e), "passed": False})
        print(f"[{i+1}/100] HATA: {e}")

    time.sleep(3)

faithfulness_df = pd.DataFrame(faithfulness_results)
faithfulness_df.to_csv("faithfulness_results.csv", index=False)
print(f"\nOrtalama Skor: {faithfulness_df['score'].mean():.2f}")
print(f"Başarılı: {faithfulness_df['passed'].sum()}/100")

Output()

**************************************************

Faithfulness Verbose Logs

**************************************************

Truths (limit=None):
[
    "Açýlma Saati: 14.00",
    "Kapanma Saati: 22.20"
] 
 
Claims:
[
    "Bu metin, Türkiye Büyük Millet Meclisi (TBMM) oturumunun bir bölümünü içeriyor.",
    "TBMM'nin 82'nci Birleşiminin Altıncı Oturumu'nun açılışı ve bu oturma sırasında tartışılan kanun teklifleri 
hakkında bilgi verilmektedir.",
    "Görüşülen konular arasında, 'Sosyal Yardım Veri Tabanı Kanunu' ve 'Sosyal Ağ Sağlayıcısı Düzenlemesi' gibi 
önemli konu başlıkları yer almaktadır.",
    "Bu konularda hükûmet yetkilileri ve milletvekilleri arasındaki tartışmalar hakkında bilgi verilmektedir.",
    "Genel olarak, TBMM'nin 82'nci Birleşiminin Altıncı Oturumu'nda kanun teklifleri ve diğer konular hakkında 
discussionlar yapılmakta ve fikirlerin değiştirilmeye çalışılmaktadır."
] 
 
Verdicts:
[
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "no",
        "reason": "The claim mentions the 82nd session of TBMM, but the retrieval context only provides information
about the start and end times of a single session."
    },
    {
        "verdict": "idk",
        "reason": "The claim mentions 'Sosyal Yard\u0131m Veri Taban\u0131 Kanunu' and 'Sosyal A\u011f 
Sa\u011flay\u0131c\u0131s\u0131 D\u00fczenlemesi', which are not mentioned in the retrieval context. However, this 
does not necessarily mean they were not discussed."
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "no",
        "reason": "The claim mentions that discussions and debates took place about kanun teklifleri (bill 
proposals) and other topics, but the retrieval context only provides information about the start and end times of a
session."
    }
]
 
Score: 0.6
Reason: The score is 0.60 because the actual output deviates from the retrieval context in two instances: it 
incorrectly references a specific session (82nd) not supported by the provided information, and it inaccurately 
mentions discussions about bill proposals when the context only covers session times.

...

Faithfulness Verbose Logs

**************************************************

Truths (limit=None):
[
    "28'inci Dönem Dördüncü Yasama Yılı",
    "1 Ekim 2025 Çarşamba günü"
] 
 
Claims:
[
    "Bu metin, Türkiye Büyük Millet Meclisinin (TBMM) Genel Kurulunun açılış konuşmasıdır.",
    "Cumhurbaşkanı Recep Tayyip Erdoğan tarafından verilen konulardan bahseden metinde, Türkiye'nin ekonomik ve 
siyasi gelişmeleri hakkında bilgi verilmektedir.",
    "Cumhurbaşkanı Erdoğan, Türkiye'nin son yirmi üç yılda alınmış büyük mesafeyi Türkiye Yüzyılı'na yakınsayacak 
ve Türkiye'ye dar gelmeyecek yeni bir anayasayla taçlandıracağına inandığını belirtmiştir.",
    "Türkiye'nin imkânlarını, potansiyelini, zenginliklerini daha da artırmak için koşturduğunu ve ülkenin 86 
milyon vatandaşının ihtiyaçları için seferber edildiğini vurgulamıştır.",
    "TBMM'nin gündemindeki işlerin sıralamasında değişikliğe gidileceği ve bazı konuların daha öncelikli olarak ele
alınacağı belirtilmiştir.",
    "Denetim konuları geçirilmeyecek ve gündemin 'Kanun Teklifleri ile Komisyonlardan Gelen Diğer İşler' kısmında 
bulunan işlerin görüştürüleceği, ancak siyasi parti grupları adına yapılacak konuşmaların süresinin en fazla 2 kişi
tarafından kullanılacağı açıklanmıştır.",
    "Genel Kurulun 28'inci Yasama Dönemi Dördüncü Yasama Yılı açılış konuşmasıyla ilgili öneri sunulmuştur ve 
Danışma Kurulu önerisi kabul edilmiştir."
] 
 
Verdicts:
[
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "no",
        "reason": "The text does not mention the economic and political developments of Turkey, it only mentions 
the topics given by President Recep Tayyip Erdo\\u00fcan."
    },
    {
        "verdict": "idk",
        "reason": "There is no information in the retrieval context to confirm or deny this claim. It seems like an
opinion or a statement made by President Erdogan."
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "no",
        "reason": "The text does not mention any changes in the order of business in the TBMM's agenda, it only 
mentions that some topics will be prioritized over others."
    },
    {
        "verdict": "idk",
        "reason": "There is no information in the retrieval context to confirm or deny this claim. It seems like a 
statement made by President Erdogan about his intentions for the future."
    },
    {
        "verdict": "yes",
        "reason": null
    }
]
 
Score: 0.7142857142857143
Reason: The score is 0.71 because the actual output deviates from the retrieval context by mentioning specific 
topics mentioned by President Recep Tayyip Erdogan but omitting broader economic and political developments of 
Turkey, as well as not detailing changes in the TBMM's agenda.

======================================================================

[82/100] Score: 0.71 | The score is 0.71 because the actual output deviates from the retrieval context by mentioning specific topics mentioned by President Recep Tayyip Erdogan but omitting broader economic and political developments of Turkey, as well as not detailing changes in the TBMM's agenda.

Ortalama Skor: 0.59
Başarılı: 14/100


In [18]:
import os
import time
import json
import pandas as pd
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval
from deepeval.models import OllamaModel

os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"

df = pd.read_csv("/summaries_turkish.csv")

metric = GEval(
    name="Action Item Quality",
    criteria="""
        Evaluate the action items extracted from the meeting transcript:
        1. Are all action items actually mentioned in the transcript?
        2. Are the descriptions clear and actionable?
        3. Are assignees only real speakers from the transcript (not invented names)?
        4. Are any important action items missed?
    """,
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    model=OllamaModel(model="llama3.1"),
    threshold=0.3
)

geval_results = []

for i, row in df.head(82).iterrows():
    try:
        parsed_content = json.loads(row["generated_response"])
        if isinstance(parsed_content, dict):
            # Action items kısmını değerlendiriyoruz
            action_items = parsed_content.get("action_items", [])
            actual_output = json.dumps(action_items, ensure_ascii=False)
        else:
            actual_output = str(parsed_content)
    except (json.JSONDecodeError, TypeError):
        actual_output = str(row["generated_response"])

    test_case = LLMTestCase(
        input=row["full_text"],
        actual_output=actual_output
    )

    try:
        metric.measure(test_case)
        geval_results.append({
            "index": i,
            "score": metric.score,
            "reason": metric.reason,
            "passed": metric.is_successful()
        })
        print(f"[{i+1}/82] Score: {metric.score:.2f} | {metric.reason}")
    except Exception as e:
        geval_results.append({"index": i, "score": None, "reason": str(e), "passed": False})
        print(f"[{i+1}/82] HATA: {e}")

    time.sleep(3)

geval_df = pd.DataFrame(geval_results)
geval_df.to_csv("geval_results.csv", index=False)
print(f"\nOrtalama Skor: {geval_df['score'].mean():.2f}")
print(f"Başarılı: {geval_df['passed'].sum()}/100")

Output()

[1/82] Score: 0.00 | The text is a summary of the proceedings of the Turkish Parliament's 82nd session, specifically the sixth meeting. It includes discussions on various bills and laws, including the Social Assistance Data Bank Law and the Social Network Provider Regulation. The text provides information on the debates and discussions between government officials and parliamentarians during this meeting.


Output()

[2/82] Score: 0.00 | The text is a summary of the proceedings of the Turkish Parliament's 81st session, specifically the sixth meeting. It mentions various topics discussed by members of parliament, including a law proposal, tax issues, and environmental concerns.


Output()

...

[80/82] Score: 0.00 | Bu metin, Türkiye Büyük Millet Meclisinin (TBMM) 3'üncü Birleşiminin Üçüncü Oturumundaki gelişmeleri anlatmaktadır. Oturumda, CHP'nin "Üniversite Öğrencilerimizin Barınma, Beslenme, Yaşam Koşulları ve Burs Kredi Sistemine İlişkin Sorunlarının Tespit ve Çözüm Önerilerinin Geliştirilmesi Amacıyla Meclis Araştırması Yapılmasına Dair" önergesi sunulmuştur. Bu önerge üzerine AK PARTI Grubundan Nazım Elmas, Adalet ve Kalkınma Partisi'nin (AKP) 20 yıla yakın bir süredir üniversite öğrencileri için yaptığı düzenlemeleri anlattı.


Output()

[81/82] Score: 0.00 | The provided text is a summary of the proceedings of the Turkish Parliament (TBMM) on a specific date, detailing various speeches and discussions by members of parliament.


Output()

[82/82] Score: 0.00 | The text is a speech by the President of Turkey, Recep Tayyip Erdoğan, at the opening of the 28th Legislative Period of the Grand National Assembly of Turkey. The speech covers various topics including economic and political developments in Turkey.

Ortalama Skor: 0.01
Başarılı: 0/100


In [19]:
import json
import pandas as pd
from rouge_score import rouge_scorer

df = pd.read_csv("/summaries_turkish.csv")
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge_results = []

for i, row in df.head(100).iterrows():
    try:
        parsed_content = json.loads(row["generated_response"])
        if isinstance(parsed_content, dict):
            generated_summary = parsed_content.get("summary", str(parsed_content))
        else:
            generated_summary = str(parsed_content)
    except (json.JSONDecodeError, TypeError):
        generated_summary = str(row["generated_response"])

    reference_summary = str(row["reference_summary"])  # ground truth — MeetingBank'ta zaten var

    scores = scorer.score(target=reference_summary, prediction=generated_summary)

    rouge_results.append({
        "index": i,
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    })
    print(f"[{i+1}/100] ROUGE-1: {scores['rouge1'].fmeasure:.3f} | ROUGE-L: {scores['rougeL'].fmeasure:.3f}")

rouge_df = pd.DataFrame(rouge_results)
rouge_df.to_csv("rouge_results.csv", index=False)
print(f"\nOrtalama ROUGE-1: {rouge_df['rouge1'].mean():.3f}")
print(f"Ortalama ROUGE-2: {rouge_df['rouge2'].mean():.3f}")
print(f"Ortalama ROUGE-L: {rouge_df['rougeL'].mean():.3f}")

[1/100] ROUGE-1: 0.078 | ROUGE-L: 0.042
[2/100] ROUGE-1: 0.103 | ROUGE-L: 0.049
[3/100] ROUGE-1: 0.165 | ROUGE-L: 0.106
[4/100] ROUGE-1: 0.042 | ROUGE-L: 0.028
[5/100] ROUGE-1: 0.015 | ROUGE-L: 0.011
[6/100] ROUGE-1: 0.174 | ROUGE-L: 0.145
[7/100] ROUGE-1: 0.100 | ROUGE-L: 0.059
[8/100] ROUGE-1: 0.191 | ROUGE-L: 0.123
[9/100] ROUGE-1: 0.041 | ROUGE-L: 0.029
[10/100] ROUGE-1: 0.085 | ROUGE-L: 0.042
[11/100] ROUGE-1: 0.109 | ROUGE-L: 0.062
[12/100] ROUGE-1: 0.163 | ROUGE-L: 0.107
[13/100] ROUGE-1: 0.031 | ROUGE-L: 0.030
[14/100] ROUGE-1: 0.225 | ROUGE-L: 0.104
[15/100] ROUGE-1: 0.061 | ROUGE-L: 0.047
[16/100] ROUGE-1: 0.059 | ROUGE-L: 0.038
[17/100] ROUGE-1: 0.036 | ROUGE-L: 0.033
[18/100] ROUGE-1: 0.097 | ROUGE-L: 0.050
[19/100] ROUGE-1: 0.043 | ROUGE-L: 0.025
[20/100] ROUGE-1: 0.050 | ROUGE-L: 0.035
[21/100] ROUGE-1: 0.031 | ROUGE-L: 0.023
[22/100] ROUGE-1: 0.110 | ROUGE-L: 0.067
[23/100] ROUGE-1: 0.043 | ROUGE-L: 0.032
[24/100] ROUGE-1: 0.048 | ROUGE-L: 0.035
[25/100] ROUGE-1: 0.023 |